# Install Required Libraries

In [ ]:
!pip install numpy pandas scikit-learn tensorflow imbalanced-learn

# Import Libraries

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

import warnings
warnings.filterwarnings('ignore')

# Upload Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

# Load Dataset

In [ ]:
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print(df.head())
print(df.shape)

# Data Preprocessing

In [ ]:
df.dropna(inplace=True)

label_encoder = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = label_encoder.fit_transform(df[col])

# Feature and Target Separation

In [ ]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Feature Scaling

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Reshape Data for LSTM

In [ ]:
X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

# Simplified Enhanced Spotted Hyena Optimizer

In [ ]:
import random

population_size = 5
iterations = 5

best_units = 64
best_dropout = 0.2
best_learning_rate = 0.001
best_accuracy = 0

for i in range(iterations):

    units = random.choice([32, 64, 128])
    dropout = random.choice([0.2, 0.3, 0.4])
    learning_rate = random.choice([0.001, 0.0005])

    model = Sequential()

    model.add(LSTM(units, input_shape=(X_train.shape[1], X_train.shape[2])))
    model.add(Dropout(dropout))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    optimizer = Adam(learning_rate=learning_rate)

    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=3, batch_size=64, verbose=0)

    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    print(f'Iteration {i+1} Accuracy: {accuracy}')

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_units = units
        best_dropout = dropout
        best_learning_rate = learning_rate

# Best Hyperparameters

In [ ]:
print('Best Units:', best_units)
print('Best Dropout:', best_dropout)
print('Best Learning Rate:', best_learning_rate)

# Final E-SHO Optimized LSTM Model

In [ ]:
model = Sequential()

model.add(LSTM(best_units, input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(Dropout(best_dropout))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

optimizer = Adam(learning_rate=best_learning_rate)

model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

# Prediction

In [ ]:
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

# Performance Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('Accuracy :', accuracy)
print('Precision:', precision)
print('Recall   :', recall)
print('F1 Score :', f1)

# Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

# Plot Accuracy Graph

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'])

plt.show()

# Save Model

In [ ]:
model.save('ESHO_LSTM_DDoS_Model.h5')